# Домашнее задание 17: Регрессия с PyTorch

**Задача:** Создать и обучить регрессионную модель, аппроксимирующую функцию:

$$f(x, y) = \sin(x + 2y) \cdot \exp\left(-(2x + y)^2\right)$$

на диапазоне $x, y \in [-10, 10]$.

- 20 000 точек, сгенерированных случайно
- Разбиение: train 70% / test 15% / val 15%
- Метрика: MSE на test
- График: истинная функция vs предсказания модели

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import mean_squared_error

# Фиксируем сиды для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

# Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Генерация данных

In [ ]:
# Генерируем 20000 случайных точек (x, y) в диапазоне [-10, 10]
N = 20000
X_raw = np.random.uniform(-10, 10, (N, 2)).astype(np.float32)
x_vals = X_raw[:, 0]
y_vals = X_raw[:, 1]

# Целевая функция: z = sin(x + 2y) * exp(-(2x + y)^2)
def true_function(x, y):
    return np.sin(x + 2 * y) * np.exp(-(2 * x + y) ** 2)

z_vals = true_function(x_vals, y_vals).astype(np.float32)

print(f'X shape: {X_raw.shape}, z shape: {z_vals.shape}')
print(f'z range: [{z_vals.min():.4f}, {z_vals.max():.4f}]')

## 2. Разделение на train/test/val (70%/15%/15%)

In [ ]:
# Создаём тензорный датасет
dataset = TensorDataset(torch.from_numpy(X_raw), torch.from_numpy(z_vals).view(-1, 1))

# Размеры выборок
train_size = int(0.7 * N)   # 14000
test_size = int(0.15 * N)    # 3000
val_size = N - train_size - test_size  # 3000

train_dataset, test_dataset, val_dataset = random_split(
    dataset, [train_size, test_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}, Val: {len(val_dataset)}')

# DataLoader-ы
batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## 3. Определение модели

In [ ]:
class RegressionNet(nn.Module):
    """Нейросеть для регрессии: 2 входа (x, y) -> несколько скрытых слоёв -> 1 выход z."""
    def __init__(self, hidden_sizes=[64, 128, 128, 64]):
        super().__init__()
        layers = []
        in_features = 2
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            layers.append(nn.ReLU())
            in_features = h
        layers.append(nn.Linear(in_features, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = RegressionNet().to(device)
print(model)
print(f'\nNumber of parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Обучение модели

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

num_epochs = 500
train_losses = []
val_losses = []

best_val_loss = float('inf')
best_model_state = None
patience = 50
patience_counter = 0

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for batch_X, batch_z in train_loader:
        batch_X, batch_z = batch_X.to(device), batch_z.to(device)
        
        optimizer.zero_grad()
        pred = model(batch_X)
        loss = criterion(pred, batch_z)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_z in val_loader:
            batch_X, batch_z = batch_X.to(device), batch_z.to(device)
            pred = model(batch_X)
            loss = criterion(pred, batch_z)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)
    
    scheduler.step()
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch + 1}')
        break
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch + 1:4d}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}')

# Восстанавливаем лучшую модель
model.load_state_dict(best_model_state)
print(f'\nBest Val Loss: {best_val_loss:.6f}')

## 5. График обучения

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', alpha=0.7)
plt.plot(val_losses, label='Validation Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Оценка MSE на тестовой выборке

In [ ]:
model.eval()
all_test_X = []
all_test_z_true = []
all_test_z_pred = []

with torch.no_grad():
    for batch_X, batch_z in test_loader:
        batch_X = batch_X.to(device)
        pred = model(batch_X).cpu().numpy()
        all_test_X.append(batch_X.cpu().numpy())
        all_test_z_true.append(batch_z.numpy())
        all_test_z_pred.append(pred)

test_X = np.vstack(all_test_X)
test_z_true = np.vstack(all_test_z_true).flatten()
test_z_pred = np.vstack(all_test_z_pred).flatten()

mse_test = mean_squared_error(test_z_true, test_z_pred)
rmse_test = np.sqrt(mse_test)
print(f'Test MSE:  {mse_test:.8f}')
print(f'Test RMSE: {rmse_test:.8f}')

# Baseline MSE (предсказание средним)
baseline_mse = mean_squared_error(test_z_true, np.full_like(test_z_true, test_z_true.mean()))
print(f'Baseline MSE (mean): {baseline_mse:.8f}')
print(f'R² ≈ {1 - mse_test / baseline_mse:.4f}')

## 7. График: истинная функция vs предсказания модели

### 7.1 3D-график истинной функции и предсказаний на тестовой выборке

In [ ]:
fig = plt.figure(figsize=(16, 6))

# --- Истинная функция ---
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
sc1 = ax1.scatter(test_X[:, 0], test_X[:, 1], test_z_true, 
                   c=test_z_true, cmap='viridis', s=5, alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')
ax1.set_title('True function z = f(x, y)')
plt.colorbar(sc1, ax=ax1, shrink=0.5)

# --- Предсказания модели ---
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
sc2 = ax2.scatter(test_X[:, 0], test_X[:, 1], test_z_pred,
                   c=test_z_pred, cmap='viridis', s=5, alpha=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
ax2.set_title('Predicted function (model)')
plt.colorbar(sc2, ax=ax2, shrink=0.5)

plt.suptitle(f'Comparison: True vs Predicted\nTest MSE = {mse_test:.6f}', fontsize=14)
plt.tight_layout()
plt.show()

### 7.2 Ошибка предсказания (True - Pred)

In [ ]:
error = test_z_true - test_z_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма ошибок
axes[0].hist(error, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Error (true - pred)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Error Distribution\nMean: {error.mean():.6f}, Std: {error.std():.6f}')
axes[0].grid(True, alpha=0.3)

# Scatter: true vs pred
axes[1].scatter(test_z_true, test_z_pred, s=5, alpha=0.5, color='steelblue')
axes[1].plot([test_z_true.min(), test_z_true.max()], 
             [test_z_true.min(), test_z_true.max()], 'r--', linewidth=1.5, label='y = x')
axes[1].set_xlabel('True z')
axes[1].set_ylabel('Predicted z')
axes[1].set_title('True vs Predicted')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.3 3D график ошибки

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(test_X[:, 0], test_X[:, 1], error,
                c=error, cmap='RdBu_r', s=5, alpha=0.8)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('Error')
ax.set_title('Prediction Error over (x, y) domain')
plt.colorbar(sc, ax=ax, shrink=0.5)
plt.tight_layout()
plt.show()

## Выводы

- Модель аппроксимирует функцию $f(x, y) = \sin(x + 2y) \cdot \exp(-(2x + y)^2)$
- MSE на тестовой выборке указана выше
- Графики демонстрируют качество аппроксимации